<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab2/SPBD_Labs_mapreduce1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 - MapReduce + MRJob Library

[MrJob](https://mrjob.readthedocs.io/en/latest/) can be used to write MapReduce jobs and run them on several platforms.

Some key advantages:
+ Write **multi-step** MapReduce jobs in pure Python;
+ Test on your **local machine**;
+ Deploy jobs in several cloud plataforms of several vendors.


 <br>
 Note: The lastest version of MrJob requires python 3.12 or older.

In [ ]:
#@title Install Python3.12 + MrJob
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install python3.12 python3-pip -qq -y
!python3.12 -m pip install mrjob --break-system-packages --quiet
!pip install mrjob --quiet


##1. Word Frequency

Create a notebook program that counts the frequency of each word in a text file (for example, the novel "Os Maias").

a) No sorting required: words and their frequencies can appear in any order.

b) Sorted by word, in ascending alphabetical order.

c) Sorted by frequency, in descending order (most frequent words first).


In [ ]:
#@title Download Dataset
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

### a) Word Frequency - No sorting required

The mapper emits `(word, 1)` for every word in a line, and the reducer sums the final count per word.

In [ ]:
%%writefile word_freq_1a.py

import re
from mrjob.job import MRJob
from mrjob.step import MRStep
class MRWordFreq1a(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordFreq1a.run()


In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1a -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

In [ ]:
%%writefile word_freq_1a_comb.py
#@title Using the combiner

import re
from mrjob.job import MRJob

class MRWordFreq1aComb(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def combiner(self, word, counts):
        yield word, sum(counts)

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordFreq1aComb.run()


In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1a_comb -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

### b) Sorted by word (ascending)

In [ ]:
%%writefile word_freq_1b.py
import re
from mrjob.job import MRJob, MRStep

class MRWordFreq1b(MRJob):

    # ---------- Step 1: count words ----------
    def mapper_count(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def combiner_count(self, word, counts):      # local pre-aggregation
        yield word, sum(counts)

    def reducer_count(self, word, counts):
        yield word, sum(counts)

    # ---------- Step 2: global sort ----------
    def mapper_collect(self, word, count):
        yield None, (word, count)

    def reducer_sort(self, _, pairs):
        for word, count in sorted(pairs):
            yield word, count

    def steps(self):
        return [MRStep(mapper=self.mapper_count, combiner=self.combiner_count,reducer=self.reducer_count),
                MRStep(mapper=self.mapper_collect, reducer=self.reducer_sort)]

if __name__ == "__main__":
    MRWordFreq1b.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1b -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

### c) Sorted by frequency (descending)

In [ ]:
%%file word_freq_1c.py

import re
from mrjob.job import MRJob, MRStep

MAX_FREQ = 1000000
class MRWordCountFrequency1c(MRJob):

  def mapper_frq(self, _, line):
    for word in re.findall(r"[^\W_]+", line.lower()):
      yield word, 1

  def combiner_frq(self, word, counts ):
      yield word, sum( counts )

  def reducer_frq(self, word, counts):
      yield word, sum( counts )

  def mapper_sort(self, word, counts):
      yield "{:06}".format(MAX_FREQ-int(counts)), word

  def reducer_sort(self, frequency, words):
    for word in words:
      yield word, MAX_FREQ - int(frequency)

  def steps(self):
    return [ MRStep(mapper=self.mapper_frq, combiner=self.combiner_frq, reducer=self.reducer_frq),
            MRStep(mapper=self.mapper_sort, reducer=self.reducer_sort)
    ]

if __name__ == '__main__':
    MRWordCountFrequency1c.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1c -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10